In [0]:
storage_account = "insurancestorage01"

spark.conf.set(
"fs.azure.account.key."+storage_account+".dfs.core.windows.net",
"azure_Storage_key"
)

spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

bronze_adls_path = "abfss://bronze@"+storage_account+".dfs.core.windows.net/customer_master/"
silver_adls_path = "abfss://silver@"+storage_account+".dfs.core.windows.net/customer_master/"

In [0]:
spark.read.parquet(bronze_adls_path).printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- customer_external_ref: string (nullable = true)
 |-- agent_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- middle_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- date_of_birth: timestamp (nullable = true)
 |-- gender: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- pincode: string (nullable = true)
 |-- country: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- annual_income: decimal(38,18) (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- height_cm: decimal(38,18) (nullable = true)
 |-- weight_kg: decimal(38,18) (nullable = true)
 |-- smoker_flag: boolean (nullable = true)
 |

# SCHEMA ENFORECEMENT

In [0]:
from pyspark.sql.types import *

customer_schema = StructType([

    StructField("customer_id", LongType(), True),
    StructField("customer_external_ref", StringType(), True),
    StructField("agent_id", IntegerType(), True),

    StructField("first_name", StringType(), True),
    StructField("middle_name", StringType(), True),
    StructField("last_name", StringType(), True),

    StructField("date_of_birth", TimestampType(), True),
    StructField("gender", StringType(), True),
    StructField("marital_status", StringType(), True),

    StructField("email", StringType(), True),
    StructField("phone_number", StringType(), True),

    StructField("address", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("pincode", StringType(), True),
    StructField("country", StringType(), True),

    StructField("occupation", StringType(), True),
    StructField("employment_type", StringType(), True),

    StructField("annual_income", DecimalType(38,18), True),
    StructField("credit_score", IntegerType(), True),

    StructField("height_cm", DecimalType(38,18), True),
    StructField("weight_kg", DecimalType(38,18), True),

    StructField("smoker_flag", BooleanType(), True),
    StructField("alcohol_consumption_level", StringType(), True),
    StructField("physical_activity_level", StringType(), True),

    StructField("pre_existing_conditions", StringType(), True),
    StructField("chronic_disease_flag", BooleanType(), True),
    StructField("family_medical_history_flag", BooleanType(), True),

    StructField("stress_level_index", IntegerType(), True),
    StructField("sleep_hours_avg", DecimalType(38,18), True),

    StructField("created_at", TimestampType(), True),
    StructField("updated_at", TimestampType(), True),

    StructField("source_system", StringType(), True),
    StructField("record_version", IntegerType(), True),

    StructField("ingestion_date", DateType(), True)

])

In [0]:
customer_df = (spark.read
               .format("parquet")
               .option("recursiveFileLookup", "true")
               .schema(customer_schema)
               .load(bronze_adls_path)
               )

In [0]:
from pyspark.sql.functions import *

customer_df_silver = (
    customer_df
    .withColumn("first_name", trim(col("first_name")))
    .withColumn("last_name", trim(col("last_name")))
    .withColumn("city", trim(col("city")))
    .withColumn("state", trim(col("state")))
)

customer_df_silver = (
    customer_df_silver

    # Type casting
    .withColumn("annual_income", col("annual_income").cast("double"))
    .withColumn("height_cm", col("height_cm").cast("double"))
    .withColumn("weight_kg", col("weight_kg").cast("double"))
    .withColumn("sleep_hours_avg", col("sleep_hours_avg").cast("double"))

    # Derived fields
    .withColumn("full_name", concat_ws(" ", col("first_name"), col("middle_name"), col("last_name")))

    .withColumn(
        "age",
        floor(datediff(current_date(), col("date_of_birth")) / 365.25)
    )

    .withColumn(
        "bmi",
        col("weight_kg") / ((col("height_cm") / 100) ** 2)
    )

    .withColumn(
        "income_band",
        when(col("annual_income") < 500000, "Low")
        .when(col("annual_income") < 1000000, "Middle")
        .when(col("annual_income") < 2500000, "Upper-Middle")
        .when(col("annual_income") < 10000000, "High")
        .otherwise("Ultra High")
    )

    .withColumn(
        "health_risk_flag",
        when(
            col("smoker_flag") |
            col("chronic_disease_flag") |
            (col("bmi") > 30),
            1
        ).otherwise(0)
    )
)

In [0]:
from pyspark.sql.window import Window

window = Window.partitionBy(col("customer_id")).orderBy(col("updated_at"))

customer_df_silver = (
    customer_df_silver
    .withColumn("rnk",rank().over(window))
    .filter(col("rnk")==1)
    .drop(col("rnk"))
)

In [0]:
customer_df_silver_final = customer_df_silver.select(

    # ---- Primary Identifiers ----
    "customer_id",
    "customer_external_ref",
    "agent_id",

    # ---- Customer Identity ----
    "first_name",
    "middle_name",
    "last_name",
    "full_name",
    "gender",
    "marital_status",
    "date_of_birth",
    "age",

    # ---- Contact Information ----
    "email",
    "phone_number",

    # ---- Address ----
    "address",
    "city",
    "state",
    "pincode",
    "country",

    # ---- Employment & Financial ----
    "occupation",
    "employment_type",
    "annual_income",
    "income_band",
    "credit_score",

    # ---- Health Metrics ----
    "height_cm",
    "weight_kg",
    "bmi",
    "smoker_flag",
    "alcohol_consumption_level",
    "physical_activity_level",
    "pre_existing_conditions",
    "chronic_disease_flag",
    "family_medical_history_flag",
    "stress_level_index",
    "sleep_hours_avg",
    "health_risk_flag",

    # ---- Metadata ----
    "created_at",
    "updated_at",
    "source_system",
    "record_version",
    "ingestion_date"
)

customer_df_silver_final.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_adls_path)